# MLOps Assignment 2 — DistilBERT Goodreads Genre Classifier
**IIT Jodhpur | PGD AI Programme | MLOps**

Fine-tunes `distilbert-base-cased` on UCSD Goodreads reviews to classify books into 7 genres.  
Trained on **Kaggle** (GPU T4) with **W&B** experiment tracking and model hosted on **Hugging Face Hub**.

| Resource | Link |
|----------|------|
| GitHub | https://github.com/g25ait2033-cyber/saiharshith |
| W&B Dashboard | https://wandb.ai/g25ait2033-prom-iit-rajasthan/mlops-assignment2 |
| Hugging Face | https://huggingface.co/g25ait2033-cyber/distilbert-goodreads-genres |

In [1]:
# Task 1: Setup — Install required packages
!pip install -q transformers datasets wandb huggingface_hub scikit-learn

## Task 1 — Kaggle Setup & Secrets

In [2]:
# Load API tokens from Kaggle Secrets (Add-ons → Secrets)
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

# Read tokens from Kaggle Secrets
hf_token    = secrets.get_secret("HF_TOKEN")
wandb_token = secrets.get_secret("WANDB_API_KEY")

os.environ["HF_TOKEN"]      = hf_token
os.environ["WANDB_API_KEY"] = wandb_token

print("Secrets loaded successfully (tokens are hidden).")

Secrets loaded successfully (tokens are hidden).


In [3]:
import os, shutil

os.chdir('/kaggle/working')

# Remove stale clone if present
if os.path.exists("saiharshith"):
    shutil.rmtree('saiharshith')

!git clone https://github.com/g25ait2033-cyber/saiharshith.git
os.chdir('/kaggle/working/saiharshith')

print("Current directory:", os.getcwd())
!ls

Cloning into 'saiharshith'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.
Current directory: /kaggle/working/saiharshith
README.md


## Task 2 — Data Loading, Encoding & Model Loading

In [4]:
# Task 2: Prepare dataset — download UCSD Goodreads reviews, split, encode
import gzip, json, pickle, random, requests
import torch
from transformers import DistilBertTokenizerFast
from sklearn.metrics import accuracy_score, f1_score

label2id, id2label = {}, {}

def build_label_maps(labels):
    global label2id, id2label
    unique = sorted(set(labels))
    label2id = {lbl: idx for idx, lbl in enumerate(unique)}
    id2label  = {idx: lbl for lbl, idx in label2id.items()}
    return label2id, id2label

class MyDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1":       f1_score(labels, preds, average="weighted"),
    }

GENRE_URL_DICT = {
    "poetry":                 "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_poetry.json.gz",
    "comics_graphic":         "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_comics_graphic.json.gz",
    "fantasy_paranormal":     "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_fantasy_paranormal.json.gz",
    "history_biography":      "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_history_biography.json.gz",
    "mystery_thriller_crime": "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_mystery_thriller_crime.json.gz",
    "romance":                "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_romance.json.gz",
    "young_adult":            "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_young_adult.json.gz",
}


def load_reviews(url, head=10000, sample_size=2000):
    reviews  = []
    response = requests.get(url, stream=True)
    response.raise_for_status()
    with gzip.open(response.raw, "rt", encoding="utf-8") as fh:
        for i, line in enumerate(fh):
            if head is not None and i >= head:
                break
            reviews.append(json.loads(line)["review_text"])
    return random.sample(reviews, min(sample_size, len(reviews)))


def load_all_genres(pickle_path="genre_reviews_dict.pickle", head=10000, sample_size=2000):
    try:
        return pickle.load(open(pickle_path, "rb"))
    except FileNotFoundError:
        pass
    genre_reviews_dict = {}
    for genre, url in GENRE_URL_DICT.items():
        print(f"Downloading: {genre}")
        genre_reviews_dict[genre] = load_reviews(url, head=head, sample_size=sample_size)
    pickle.dump(genre_reviews_dict, open(pickle_path, "wb"))
    return genre_reviews_dict


def train_test_split_genres(genre_reviews_dict, reviews_per_genre=1000, train_frac=0.8):
    train_texts, train_labels, test_texts, test_labels = [], [], [], []
    for genre, reviews in genre_reviews_dict.items():
        sampled = random.sample(reviews, min(reviews_per_genre, len(reviews)))
        split = int(len(sampled) * train_frac)
        train_texts += sampled[:split]; train_labels += [genre]*split
        test_texts += sampled[split:];  test_labels += [genre]*(len(sampled)-split)
    print(f"Train: {len(train_texts)} | Test: {len(test_texts)} | Genres: {len(genre_reviews_dict)}")
    return train_texts, train_labels, test_texts, test_labels


def encode_datasets(tr_texts, tr_labels, te_texts, te_labels, tokenizer, max_length=512):
    l2i, _ = build_label_maps(tr_labels)
    tr_enc  = tokenizer(tr_texts, truncation=True, padding=True, max_length=max_length)
    te_enc  = tokenizer(te_texts, truncation=True, padding=True, max_length=max_length)
    return (
        MyDataset(tr_enc, [l2i[y] for y in tr_labels]),
        MyDataset(te_enc, [l2i[y] for y in te_labels]),
    )


# Run data pipeline
MODEL_NAME = "distilbert-base-cased"
tokenizer  = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

genre_reviews = load_all_genres()
tr_texts, tr_labels, te_texts, te_labels = train_test_split_genres(genre_reviews)
train_dataset, test_dataset = encode_datasets(tr_texts, tr_labels, te_texts, te_labels, tokenizer)

print(f"Train: {len(train_dataset)} | Test: {len(test_dataset)} | Labels: {len(id2label)}")
print("Label map:", id2label)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Downloading: poetry
Downloading: comics_graphic
Downloading: fantasy_paranormal
Downloading: history_biography
Downloading: mystery_thriller_crime
Downloading: romance
Downloading: young_adult
Train: 5600 | Test: 1400 | Genres: 7
Train: 5600 | Test: 1400 | Labels: 7
Label map: {0: 'comics_graphic', 1: 'fantasy_paranormal', 2: 'history_biography', 3: 'mystery_thriller_crime', 4: 'poetry', 5: 'romance', 6: 'young_adult'}


In [5]:
# Task 2: Load pre-trained DistilBERT for sequence classification
from transformers import DistilBertForSequenceClassification, TrainingArguments, Trainer
import wandb

NUM_LABELS = len(id2label)
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id
).to("cuda")

print(f"Model: {MODEL_NAME} | Classes: {NUM_LABELS} | Device: cuda")

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: distilbert-base-cased | Classes: 7 | Device: cuda


## Task 3 — Train on Kaggle GPU & Track with W&B

In [6]:
import wandb

WANDB_PROJECT = "mlops-assignment2"
WANDB_RUN     = "distilbert-run-kaggle"
HF_REPO       = "g25ait2033-cyber/distilbert-goodreads-genres"

wandb.login(key=os.environ["WANDB_API_KEY"])
wandb.init(
    project=WANDB_PROJECT, name=WANDB_RUN,
    config={"model": MODEL_NAME, "epochs": 3, "batch_size": 16,
            "learning_rate": 3e-5, "max_length": 512,
            "dataset": "UCSD Goodreads", "platform": "Kaggle"}
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="wandb",
    run_name=WANDB_RUN,
    learning_rate=3e-5,
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_dataset, eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()
print("Training complete.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: g25ait2033 (g25ait2033-prom-iit-rajasthan) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run dko86oem
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/saiharshith/wandb/run-20260527_164252-dko86oem
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run distilbert-run-kaggle
wandb: ⭐️ View project at https://wandb.ai/g25ait2033-prom-iit-rajasthan/mlops-assignment2
wandb: 🚀 View run at https://wandb.ai/g25ait2033-prom-iit-rajasthan/mlops-assignment

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,2.818733,2.382532,0.555000,0.542836
2,2.067063,2.215729,0.590714,0.577690
3,1.692744,2.156080,0.604286,0.606458


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Training complete.


## Task 3 (contd.) — Evaluation & W&B Artifact Upload

In [7]:
# Task 3: Evaluate on test set, log metrics, upload artifact
import json
from sklearn.metrics import classification_report

wandb.init(project=WANDB_PROJECT, id="os0kjbqd", resume="must")

eval_results = trainer.evaluate()
print("Eval:", eval_results)

# Log final metrics explicitly
wandb.log({
    "final/loss": eval_results["eval_loss"],
    "final/accuracy": eval_results["eval_accuracy"],
    "final/f1": eval_results["eval_f1"],
})

# Classification report
pred_output = trainer.predict(test_dataset)
preds = pred_output.predictions.argmax(-1).flatten().tolist()
preds_str = [id2label[p] for p in preds]

report = classification_report(te_labels, preds_str, output_dict=True)
print("\n", classification_report(te_labels, preds_str))

# Save & upload as W&B Artifact
with open("eval_report.json", "w") as f:
    json.dump(report, f, indent=2)

artifact = wandb.Artifact("eval-report", type="evaluation")
artifact.add_file("eval_report.json")
wandb.log_artifact(artifact)

wandb.finish()
print("Evaluation complete. Artifact uploaded.")

wandb: Finishing previous runs because reinit is set to 'default'.
wandb: updating run metadata
wandb: uploading history steps 12-13, summary, console lines 6-9
wandb: 
wandb: Run history:
wandb:           eval/accuracy ▁▆█
wandb:                 eval/f1 ▁▅█
wandb:               eval/loss █▃▁
wandb:            eval/runtime ▁▄█
wandb: eval/samples_per_second █▄▁
wandb:   eval/steps_per_second █▄▁
wandb:             train/epoch ▁▂▂▃▃▄▅▅▅▆▇███
wandb:       train/global_step ▁▂▂▃▃▄▅▅▅▆▇███
wandb:         train/grad_norm ▁▄▄▅▅▇▇█▆▆
wandb:     train/learning_rate ▄█▇▆▅▅▄▃▂▁
wandb:                      +1 ...
wandb: 
wandb: Run summary:
wandb:           eval/accuracy 0.60429
wandb:                 eval/f1 0.60646
wandb:               eval/loss 2.15608
wandb:            eval/runtime 12.9191
wandb: eval/samples_per_second 108.367
wandb:   eval/steps_per_second 1.703
wandb:              total_flos 2225650736332800.0
wandb:             train/epoch 3
wandb:       train/global_step 525
wandb:      

Eval: {'eval_loss': 2.1560795307159424, 'eval_accuracy': 0.6042857142857143, 'eval_f1': 0.6064582982624367, 'eval_runtime': 12.7834, 'eval_samples_per_second': 109.517, 'eval_steps_per_second': 1.721, 'epoch': 3.0}


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



                         precision    recall  f1-score   support

        comics_graphic       0.83      0.79      0.81       200
    fantasy_paranormal       0.48      0.48      0.48       200
     history_biography       0.68      0.62      0.65       200
mystery_thriller_crime       0.54      0.60      0.57       200
                poetry       0.78      0.83      0.80       200
               romance       0.65      0.55      0.60       200
           young_adult       0.32      0.34      0.33       200

              accuracy                           0.60      1400
             macro avg       0.61      0.60      0.61      1400
          weighted avg       0.61      0.60      0.61      1400



wandb: uploading artifact eval-report; updating run metadata
wandb: uploading artifact eval-report
wandb: uploading history steps 22-22, summary, console lines 5-18
wandb: 
wandb: Run history:
wandb:           eval/accuracy ▁
wandb:                 eval/f1 ▁
wandb:               eval/loss ▁
wandb:            eval/runtime ▁
wandb: eval/samples_per_second ▁
wandb:   eval/steps_per_second ▁
wandb:          final/accuracy ▁
wandb:                final/f1 ▁
wandb:              final/loss ▁
wandb:           test/accuracy ▁
wandb:                      +7 ...
wandb: 
wandb: Run summary:
wandb:           eval/accuracy 0.60429
wandb:                 eval/f1 0.60646
wandb:               eval/loss 2.15608
wandb:            eval/runtime 12.7834
wandb: eval/samples_per_second 109.517
wandb:   eval/steps_per_second 1.721
wandb:          final/accuracy 0.60429
wandb:                final/f1 0.60646
wandb:              final/loss 2.15608
wandb:       huggingface_model https://huggingface....
wandb:    

Evaluation complete. Artifact uploaded.


## Task 4 — Save Model & Push to Hugging Face Hub

In [8]:
# Task 4: Push model + tokenizer to Hugging Face Hub
from huggingface_hub import login

trainer.save_model("./distilbert-goodreads-genres")
tokenizer.save_pretrained("./distilbert-goodreads-genres")

login(token=os.environ["HF_TOKEN"])
model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

# Log HF URL in W&B summary
hf_url = f"https://huggingface.co/{HF_REPO}"
if wandb.run is not None:
    wandb.run.summary["huggingface_model"] = hf_url
else:
    RUN_ID = "os0kjbqd"
    ENTITY = "g25ait2033-prom-iit-rajasthan"
    api = wandb.Api()
    run = api.run(f"{ENTITY}/{WANDB_PROJECT}/{RUN_ID}")
    run.summary["huggingface_model"] = hf_url
    run.update()

print(f"Model pushed: {hf_url}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


README.md:   0%|          | 0.00/985 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Model pushed: https://huggingface.co/g25ait2033-cyber/distilbert-goodreads-genres


## Inference Demo

In [9]:
# Quick inference using the published HF model
from transformers import pipeline

classifier = pipeline("text-classification", model=HF_REPO)

samples = [
    "The detective found a fingerprint on the cold brass handle.",
    "Their eyes met across the crowded ballroom and she felt her heart race.",
    "In the quiet of the night, the soul speaks in rhyme.",
    "The 19th-century industrial revolution reshaped the global economy.",
    "The dragon unfurled its wings above the ancient stone castle.",
]

for text, res in zip(samples, classifier(samples)):
    print(f"{text[:55]:57s} → {res['label']:25s} ({res['score']:.3f})")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/329 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The detective found a fingerprint on the cold brass han   → mystery_thriller_crime    (0.838)
Their eyes met across the crowded ballroom and she felt   → poetry                    (0.393)
In the quiet of the night, the soul speaks in rhyme.      → poetry                    (0.878)
The 19th-century industrial revolution reshaped the glo   → history_biography         (0.584)
The dragon unfurled its wings above the ancient stone c   → fantasy_paranormal        (0.446)


---
## Results

| Metric | Score |
|--------|-------|
| Accuracy | 0.60286 |
| F1 Score | 0.59748 |
| Eval Loss | 2.24548 |

| Resource | Link |
|----------|------|
| GitHub | https://github.com/g25ait2033-cyber/saiharshith |
| W&B Dashboard | https://wandb.ai/g25ait2033-prom-iit-rajasthan/mlops-assignment2 |
| Hugging Face | https://huggingface.co/g25ait2033-cyber/distilbert-goodreads-genres |